# ESS.112 and telescope elevation study


## References

- [Times Square: Image Quality Nightly Report: IQ, AOS, and EAS](https://usdf-rsp.slac.stanford.edu/times-square/github/lsst-sitcom/ts_aos_analysis/notebooks/nightly_report/nightly_report_aos_eas?dayobs=20250827&ts_hide_code=1)
- [GitHub: Image Quality Nightly Report: IQ, AOS, and EAS](https://github.com/lsst-sitcom/ts_aos_analysis/blob/develop/notebooks/nightly_report/nightly_report_aos_eas.ipynb)
- [SITCOM-2212 Investigate possible correlation between temperature from ESS:112 and telescope elevation](https://rubinobs.atlassian.net/browse/SITCOM-2212)
- [SITCOM-2079 Traceback and model for the available temperature sensors during ComCam](https://rubinobs.atlassian.net/browse/SITCOM-2079)
- [LTS-192 M2 Electronics Design Document](https://docushare.lsstcorp.org/docushare/dsweb/Get/Document-54702/LTS-192%20M2%20Electronics%20Design%20Document.pdf)


## Introduction

I expect this notebook to be a little different from other's I have been working on.  
Instead of being a notebook where you simply execute all the cells,  
I expect it to be more like a study.  
Let's see how it goes.

The main challenge with this study is that the temperatures are different every day.  
So we cannot make a simple and direct correlation between the temperature measured on ESS:112 and the telescope elevation.  
Let me start then my analysis be querying the telemetry displayed in [Nightly Report EAS in Times Square].  
Then, I will consider the data from a full day obs for now.  
I will create a correlation matrix using binned data with the min and max values groupy by each half second.
  
[Nightly Report EAS in Times Square]: https://usdf-rsp.slac.stanford.edu/times-square/github/lsst-sitcom/ts_aos_analysis/notebooks/nightly_report/nightly_report_aos_eas?dayobs=20250827&ts_hide_code=1


## Initial Dataset

As mentioned above, let's start with a single day obs and let's start with the telemetry in [Nightly Report EAS in Times Square].

I started some analysis and the temperatures during the day pollute lots of what we are looking for.  
So, since I want to make my life easy, I will start my analysis looking for nights where we were totally on sky. 
After that, I will select the timestamps between the end and the beginning of astronomical twilights.  
This is to minimize the effects of temperature changes in my analysis.

In [ ]:
day_obs = 20250827

In [ ]:
import asyncio
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

from astropy.time import Time

from lsst_efd_client import EfdClient
from lsst.summit.utils.efdUtils import (
    getEfdData,
    getDayObsEndTime,
    getDayObsForTime,
    getDayObsStartTime,
    makeEfdClient,
)

In [ ]:
# Create an instance of the EfdClient
efd_client = makeEfdClient()

# Gather the start and end time of our analysis
start_time = getDayObsStartTime(day_obs)
end_time = getDayObsEndTime(day_obs)

print(f"This initial analysis will contain data from:\n"
      f"  {start_time} to {end_time}.\n")

### Select nights fully on sky

I have been going back and forth in this analysis.  
As I look at the plots, I believe it is hard to evaluate what affects what if we include the Sun in the equation.  
I want to consider the scenario where the dome is partially thermalized to make analysis easier.  

For this work, I will use the [ConsDB_visits_metadata] notebook in Times Square.  
The code seems to have exactly what I need. 

[ConsDB_visits_metadata]: https://usdf-rsp.slac.stanford.edu/times-square/github/lsst/schedview_notebooks/nightly/ConsDB_visits_metadata?day_obs_min=20250620&day_obs_max=20250820&instrument=lsstcam&ts_hide_code=1

### Select night time



### ESS:112 M2; RPi with sticker 1

The [ESS:112](https://github.com/lsst-ts/ts_config_ocs/blob/49e7dd64b62415685f28b8bc2f179a3633c05059/ESS/v8/_init.yaml#L227) is one of our favorite sensors that we use to estimate the temperature inside the dome.  
This is the sensor in which we believe that is measuring temperature gradients depending on the elevation angle. 

Previous analysis showed that the average sampling is about 1.4 seconds.  
This means that we probably don't need to subsample this dataset.  
Querying 24h of data takes around 6 seconds.

In [ ]:
ess_112_df = await efd_client.select_time_series(
    topic_name="lsst.sal.ESS.temperature", 
    fields=["temperatureItem0"], 
    start=start_time, 
    end=end_time, 
    index=112
)

In [ ]:
ess_112_df.plot(title="Inside Temperature - ESS:112", figsize=(10, 3))
plt.grid(":", alpha=0.2)

We can see lots of spikes in this data.  
This is probably what Elana is looking for. 

Part of the analysis is to look for correlation between these temperatures and the elevation angle.  
We can plot both together to see if the peaks have correlation with any elevation change.  
Let's do this.

In [ ]:
query = efd_client.build_time_range_query(
    "lsst.sal.MTMount.elevation",
    ["mean(actualPosition)"],
    start_time,
    end_time,
) + "GROUP BY time(30s)"

elevation_df = await efd_client.influx_client.query(query)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

fig.suptitle("ESS:112 Temperature and Telescope Elevation")
ax.plot(ess_112_df, color="C0", label="ESS:112 Temperature")
ax.grid(":", alpha=0.2)
ax.set_ylabel("Temperature [deg C]")
ax.legend(loc="upper right")

ax2 = ax.twinx()
ax2.plot(elevation_df, color="C1", label="Telescope Elevation")
ax2.set_ylabel("Elevation Angle [deg]")
ax2.legend(loc="lower left")

plt.show()

Maybe. A true maybe. There are some spikes and some oscillations that seem to correlate. 

 - [ ] Select nights fully on sky  
 - [ ] Select time when we are on sky  

### ESS:301 Weather Tower

[ESS:301] can give us some reference about the environment conditions.  
We know, by experience, that the temperature inside and outsite the dome can be very different.  
I might drop this one later if we decide to focus more in our analysis.  
Remember that the goal here is to try to identify a vertical gradient inside the dome.

The sampling average here is about 3.8 seconds.  
You can easily see that we will have to do some data-processing if we want to do our correlation matrix. 

[ESS:301]: https://github.com/lsst-ts/ts_config_ocs/blob/49e7dd64b62415685f28b8bc2f179a3633c05059/ESS/v8/_init.yaml#L480

In [ ]:
ess_301_df = await efd_client.select_time_series(
    topic_name="lsst.sal.ESS.temperature", 
    fields=["temperatureItem0"], 
    start=start_time, 
    end=end_time, 
    index=301
)

In [ ]:
ess_301_df.plot(title="Outside Temperature - Weather Tower", figsize=(10, 3))
plt.grid(":", alpha=0.2)

Here you can clearly see the effect of the sun in the temperature outside.  
There is some fluctuation, but the overal trend is very clear.  
This is typical of a dataset that is more robust and insensitive to other parameters, as it should be.

### MTM2.temperature

The MTM2 CSC have internal temperature sensors as well.  
The [MTM2.temperature] contains 12 channels corresponding to 12 sensors inside the cell.  
Based on a quick conversation with Elana, it seems that column `ring5` corresponds to sensor 
that is mounted temporarily outside the m2 cell.  
This is why we can also use it to track ambient temperature. 

The sampling for this dataset has an average of 0.05 seconds.  
This is much more than what we need.  
A query containing 24h of telemetry can take almost five minutes to be completed.  
Considering the time scale of the telemetry above, 
and considering that we are still in an exploratory phase, let's do a sampling of half minute.

[MTM2.temperature]: https://ts-xml.lsst.io/sal_interfaces/MTM2.html#temperature

In [ ]:
# First, let's build the query
query = efd_client.build_time_range_query(
    "lsst.sal.MTM2.temperature",
    [f"mean(ring{i}) as mean_ring_{i}" for i in range(12)],
    start_time,
    end_time,
) + "GROUP BY time(30s)"

# And then we query it, this takes a long time
mtm2_temp_df = await efd_client.influx_client.query(query)

<br>
Just because I am curious, let's have a look at the data considering all the channels.

In [ ]:
colors = ["cornflowerblue", "royalblue", "deepskyblue", "navy",
          "limegreen", "green", "olive", "olivedrab",
          "black", "silver", "rosybrown", "darkgoldenrod"]
mtm2_temp_df.plot(title="M2 Temperatures", figsize=(10, 5), color=colors)
plt.legend(ncols=3, loc="lower right")
plt.grid()

It calls my attention that `ring5` is the topic that we are using.  
It seems sensitive to "something" else. 

All the other temperature channels have a clear trend.  
Some of them show some oscillation (like ring 6 and 